In [1]:
import pandas as pd
import numpy as np

## Load cleaned poker decisions
## Relative path allows the notebook to run on Windows or Linux
filepath = "parquet/decisions/handhq/ABS-2009-07-01_2009-07-23_1000NLH_OBFU/10/part-0000.parquet"
df = pd.read_parquet(
    filepath,
    columns=["hand_id", "street", "to_call", "action"]
)

## Identify whether the player is facing a bet
df["facing_bet"] = df["to_call"] >0

## Display the first five observations
print(df.head())


   hand_id  street  to_call  action  facing_bet
0        0       0     10.0       2        True
1        0       0     47.5       0        True
2        0       0     47.5       0        True
3        0       0     47.5       0        True
4        0       0     42.5       0        True


In [2]:
## Separate poker hands into training and testing sets
hand_ids = df["hand_id"].unique()

##shuffle the hand IDs for a random split
np.random.seed(42)## fix random seed so training/testing splits are reproducible
np.random.shuffle(hand_ids)

## Use 80% for training and 20% for testing
cutoff = int(len(hand_ids)*0.8)

train = df[df["hand_id"].isin(hand_ids[:cutoff])]##contains decisions to estimate action probabilities
test = df[df["hand_id"].isin(hand_ids[cutoff:])]##contains decisions to measure accuracy

##check the number of decisions in each set
print("Training decisions:", len(train))
print("Testing decisions:", len(test))

Training decisions: 334007
Testing decisions: 83145


In [3]:
##Poker stage labels
street_labels ={
    0: "Pre-flop",
    1: "Flop",
    2: "Turn",
    3: "River"
}

##Poker action labels
action_labels = {
    0: "Fold",
    1: "Check/call",
    2: "Bet/Raise"
}


In [4]:
##Counts how often each action occurs for each poker situation
counts = pd.crosstab([train["street"], train["facing_bet"]],train["action"])

##Include all three possible actions
counts = counts.reindex(columns=[0,1,2], fill_value=0)

##Bayesian update: apply Dirichlet prior (1,1,1)
alpha = counts + 1

##Calculate posterior mean probabilities
##alpha.sum(axis=1)calculates the total updated count for each situation
##Dividing each count by its row total gives the estimated probability of each action
probabilities = alpha.div(alpha.sum(axis=1),axis=0)

##Replace numeric labels in the displayed table
labeled_probabilities = probabilities.rename(
    index=street_labels,
    level="street"
)

labeled_probabilities = labeled_probabilities.rename(
    columns=action_labels
)
print (labeled_probabilities)

action                   Fold  Check/call  Bet/Raise
street   facing_bet                                 
Pre-flop False       0.002740    0.760031   0.237228
         True        0.656108    0.137500   0.206393
Flop     False       0.000404    0.609603   0.389992
         True        0.524156    0.348256   0.127588
Turn     False       0.000091    0.631047   0.368862
         True        0.493611    0.405570   0.100819
River    False       0.000428    0.638550   0.361022
         True        0.546413    0.367156   0.086430


In [5]:
## Select the most likely action for each poker situation
##idxmax is a pandas method that finds the label associated with the largest value
most_likely = probabilities.idxmax(axis=1).reset_index(name="prediction")

## Match predictions to the testing data
##pandas method that combines 2 tables with matching cols
results = test.merge(most_likely, on=["street", "facing_bet"], how="left")

##Calculate prediction accuracy
accuracy = (results["prediction"]==results["action"]).mean()

##Baseline: always predict the most common training action
##train["action"] — Selects the actual poker actions in our training dataset
##.value_counts() — Counts how frequently each action occurs
##.idxmax() — Returns the action code with the highest count
baseline_action = train["action"].value_counts().idxmax()
baseline_accuracy = (test["action"]==baseline_action).mean()

print("Bayesian model accuracy:", round(accuracy*100,2), "%")
print("Baseline accuracy:", round(baseline_accuracy*100, 2),"%")

##Create table of actual versus predicted actions
table = pd.crosstab(
    results["action"],
    results["prediction"],
    rownames=["Actual"],
    colnames=["Predicted"]
)

##Include all three actions, even those never predicted
table = table.reindex(
    index=[0,1,2],
    columns=[0,1,2],
    fill_value=0
)

print (table.rename(
    index=action_labels,
    columns=action_labels
))

Bayesian model accuracy: 63.42 %
Baseline accuracy: 47.8 %
Predicted    Fold  Check/call  Bet/Raise
Actual                                  
Fold        39735          10          0
Check/call  10822       12996          0
Bet/Raise   12132        7450          0


In [6]:
import os
print (os.getcwd())

c:\Users\kirst\Desktop\Dartmouth\phh-dataset


In [8]:
## ===============================================================
## Inspect the Parquet Dataset
## ===============================================================

## Import libraries for locating files and reading Parquet metadata
from pathlib import Path
import pyarrow.parquet as pq

##Find all Parquet files in the poker decisions dataset
files = list(Path("parquet/decisions/handhq").rglob("*.parquet"))

##Open first file to check structure
sample = pq.ParquetFile(str(files[0]))

##Display the file location, column names, and number of rows
print("First file:", files[0])
print("Columns:", sample.schema_arrow.names)
print("Rows:", sample.metadata.num_rows)


First file: parquet\decisions\handhq\PTY-2009-07-01_2009-07-23_600NLH_OBFU\6\part-0000.parquet
Columns: ['hand_id', 'street', 'seat', 'n_players', 'n_active', 'big_blind', 'pot', 'to_call', 'stack', 'max_bet', 'n_raises', 'hole1', 'hole2', 'flop1', 'flop2', 'flop3', 'turn', 'river', 'action', 'amount']
Rows: 699897


In [13]:
## ===============================================================
## Verify Dataset Consistency
## ===============================================================

#Define columns required for our Bayesian model
required = {"hand_id", "street", "to_call", "action"}

##Store files with missing columns
missing_files = []

##Track data types to check consistency in files
column_types = {col: set() for col in required}

##Check info of each Parquet file
for file in files:

    ##Read column info without loading dataset
    schema = pq.ParquetFile(str(file)).schema_arrow

    ##Identify missing required columns
    missing = required - set(schema.names)

    ##Record files with missing required columns
    if missing:
        missing_files.append((str(file), sorted(missing)))

    else:
        ##Record data type of each required column
        for col in required:
            column_types[col].add(str(schema.field(col).type))

## ===============================================================
## Display Verification Results
## ===============================================================


print("Files checked:", len(files))
print("Files missing required columns:", len(missing_files))
print("Column_types:", column_types)
print("Examples of missing columns:", missing_files[:3])



Files checked: 232
Files missing required columns: 0
Column_types: {'hand_id': {'int64'}, 'action': {'int8'}, 'street': {'int8'}, 'to_call': {'float'}}
Examples of missing columns: []


In [19]:
## ===============================================================
## Bayesian Model - Full Poker Dataset
## ==============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

##Find all Parquet files
files = sorted(Path("parquet/decisions/handhq").rglob("*.parquet"))

##Store action counts for training and testing
##Rows represent street codes, facing-bet status, and actions
train_counts=np.zeros((256,2,3), dtype=np.int64)
test_counts=np.zeros((256, 2,3), dtype=np.int64)

total_rows=0

##Process each file without loading the entire dataset
for file in files:

    parquet_file = pq.ParquetFile(str(file))

##Read 4 columns needed by our model
    for batch in parquet_file.iter_batches(
        batch_size=250000,
        columns=["hand_id", "street", "to_call", "action"]
    ):
        df = batch.to_pandas()

        ##Extract the input features and target action
        street = df["street"].to_numpy(dtype=np.int16)
        facing_bet = (df["to_call"].to_numpy()>0).astype(np.int16)
        action = df["action"].to_numpy(dtype=np.int16)

        ##Confirm 3 action categories are correct
        if not np.isin(action, [0,1,2]).all():
            raise ValueError(f"Unexpected action code in {file}")

        ##Assign hands to training or testing
        ##Hash keeps repeated hand IDs in same set
        hand_hash = pd.util.hash_array(df["hand_id"].to_numpy())
        is_training = (hand_hash % 5)!=0

        ## Combine the feature values into counting categories
        category = (street + 128) * 6 + facing_bet * 3 + action

        ## Count training actions in each category
        train_counts += np.bincount(
            category[is_training], minlength=1536
        ).reshape(256, 2, 3)
        
        ## Count testing actions in each category
        test_counts += np.bincount(
            category[~is_training], minlength=1536
        ).reshape(256, 2, 3)

        total_rows += len(df)

## Display progress after processing all files
print("Files processed:", len(files))
print("Total decisions processed:", total_rows)

## ===============================================================
## Calculate Bayesian Probabilities
## ===============================================================

## Add the Dirichlet prior (1, 1, 1)
alpha = train_counts + 1

## Calculate posterior probabilities for each action
probabilities = alpha / alpha.sum(axis=2, keepdims=True)

## Predict the action with the highest probability
predicted_action = probabilities.argmax(axis=2)

## ===============================================================
## Evaluate the Model
## ===============================================================

## Build the confusion matrix from testing counts
confusion = np.zeros((3, 3), dtype=np.int64)

for s in range(256):
    for facing in range(2):
        prediction = predicted_action[s, facing]
        confusion[:, prediction] += test_counts[s, facing]

## Calculate accuracy
training_total = train_counts.sum()
testing_total = test_counts.sum()

accuracy = np.trace(confusion) / testing_total

## Compare against the most common training action
most_common_action = train_counts.sum(axis=(0, 1)).argmax()
baseline_accuracy = (
    test_counts.sum(axis=(0, 1))[most_common_action]
    / testing_total
)

## Display the results
action_names = ["Fold", "Check/call", "Bet/Raise"]

print("Files processed:", len(files))
print("Total decisions:", total_rows)
print("Training decisions:", training_total)
print("Testing decisions:", testing_total)
print("Bayesian model accuracy:", round(accuracy * 100, 2), "%")
print("Baseline accuracy:", round(baseline_accuracy * 100, 2), "%")

print("\nConfusion Matrix:")
print(pd.DataFrame(
    confusion,
    index=pd.Index(action_names, name="Actual"),
    columns=pd.Index(action_names, name="Predicted")
))



Files processed: 232
Total decisions processed: 174164960
Files processed: 232
Total decisions: 174164960
Training decisions: 139312471
Testing decisions: 34852489
Bayesian model accuracy: 68.14 %
Baseline accuracy: 52.52 %

Confusion Matrix:
Predicted       Fold  Check/call  Bet/Raise
Actual                                     
Fold        18297649        6234          0
Check/call   4698713     5451309          0
Bet/Raise    3610555     2788029          0


In [21]:

## ===============================================================
## Evaluate Model Performance by Action
## ===============================================================

## Calculate recall for each action category
recall = np.diag(confusion) / confusion.sum(axis=1)

## Display recall percentages
for action_name, score in zip(action_names, recall):
    print(f"{action_name} recall: {score * 100:.2f}%")

## Display the number of predictions for each action
print("\nPredictions by action:")
for action_name, count in zip(action_names, confusion.sum(axis=0)):
    print(f"{action_name}: {count:,}")


Fold recall: 99.97%
Check/call recall: 53.71%
Bet/Raise recall: 0.00%

Predictions by action:
Fold: 26,606,917
Check/call: 8,245,572
Bet/Raise: 0
